<a href="https://colab.research.google.com/github/AashishGahtori/Data_Science_Assignments/blob/main/DSA_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import pandas as pd



1.Which movie made the highest profit? Who were its producer and director? Identify the actors in that film.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv('imdb_data.csv')
df.head()

In [ ]:
df['profit'] = df['revenue'] - df['budget']


row = df.loc[df['profit'].idxmax()]
print("Title : ", row['title'])

In [ ]:
import ast
crew = ast.literal_eval(row['crew'])
director = [i['name'] for i in crew if i['job'] == 'Director']
print("Director : ", director)


In [ ]:
producer = [i['name'] for i in crew if i['job'] == 'Producer']
print("Producer : ", producer)

In [ ]:

cast = ast.literal_eval(row['cast'])
actors = [i['name'] for i in cast]
print("Actors : " , actors)

2.This data has information about movies made in different languages. Which language has the highest average ROI (return on investment)?

In [ ]:
df['ROI'] = df['profit'] / df['budget']
valid_roi = df[(df['budget'] > 0) & (df['ROI'] != np.inf) & (df['ROI'].notnull())]
ans = valid_roi.groupby('original_language')['ROI'].mean().sort_values(ascending=False)
print(ans.head(1))


3.Find out the unique genres of movies in this dataset

In [ ]:
genres_set = set()

for g in df['genres']:
    if pd.notna(g):
        genres = ast.literal_eval(g)
        for i in genres:
            genres_set.add(i['name'])

print(genres_set)


4.Make a table of all the producers and directors of each movie. Find the top 3 producers who have produced movies with the highest average RoI?

In [ ]:
rows = []

for _, r in valid_roi.iterrows():
    if pd.notna(r['crew']):
        crew = ast.literal_eval(r['crew'])
        producers = [i['name'] for i in crew if i['job']=='Producer']

        for p in producers:
            rows.append([p, r['ROI']])

temp = pd.DataFrame(rows, columns=['producer','ROI'])

print(temp.groupby('producer')['ROI'].mean().sort_values(ascending=False).head(3))


5.Which actor has acted in the most number of movies? Deep dive into the movies, genres and profits corresponding to this actor

In [ ]:

rows = []

for _, r in df.iterrows():
    if pd.notna(r['cast']):
        cast = ast.literal_eval(r['cast'])
        for i in cast:
            rows.append([i['name'], r['profit'], r['genres']])

temp = pd.DataFrame(rows, columns=['actor','profit','genres'])

top_actor = temp['actor'].value_counts().idxmax()
print("Top Actor:", top_actor)

actor_data = temp[temp['actor'] == top_actor]

print("Total Movies:", len(actor_data))
print("Avg Profit:", actor_data['profit'].mean())

# Deep dive into genres
actor_genres = set()
for g_list in actor_data['genres']:
    if pd.notna(g_list):
        genres = ast.literal_eval(g_list)
        for g in genres:
            actor_genres.add(g['name'])

print("Genres:", list(actor_genres))

6.Top 3 directors prefer which actors the most?

In [ ]:
rows = []

for _, r in df.iterrows():
    if pd.notna(r['crew']) and pd.notna(r['cast']):
        crew = ast.literal_eval(r['crew'])
        cast = ast.literal_eval(r['cast'])

        director = [i['name'] for i in crew if i['job']=='Director']
        director = director[0] if director else None

        for actor in cast:
            rows.append([director, actor['name']])

temp = pd.DataFrame(rows, columns=['director','actor'])

top_directors = temp['director'].value_counts().head(3).index

for d in top_directors:
    print("\nDirector:", d)
    print(temp[temp['director']==d]['actor'].value_counts().head(3))